<a href="https://colab.research.google.com/github/henryhkb/ecg-vision-app/blob/main/ECG_Vision_04_Phone_Photos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# drive: connects Colab to your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# os: file paths | numpy: numbers | pandas: tables | matplotlib: figures and the PDF
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# PdfPages: saves several figures as pages of one PDF | Image: opens PNG files
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image
# files.download: sends a file from Colab to your Mac
from google.colab import files

# Paths, same as before
PROJECT_DIR = '/content/drive/MyDrive/ECG_Vision_App'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
IMG_DIR = '/content/ecg_images'

# Unzip the ECG images and load the labels
!unzip -q -n {DATA_DIR}/ecg_images.zip -d /content
data = pd.read_csv(os.path.join(DATA_DIR, 'af_vs_normal_labels.csv'), index_col='ecg_id')
test_table = data[data['split'] == 'test']
print('Test ECGs:', len(test_table), '| AF:', (test_table['label'] == 1).sum())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Test ECGs: 967 | AF: 152


In [2]:
# Pick 20 AF and 20 normal ECGs from the TEST set (labels known, never used for decisions)
# random_state=0 fixes the choice, so the same 40 are picked every time you run this
af = test_table[test_table['label'] == 1].sample(20, random_state=0)
normal = test_table[test_table['label'] == 0].sample(20, random_state=0)

# Mix them, so AF and normal are not grouped together, and give each a neutral code P01 ... P40
chosen = pd.concat([af, normal]).sample(frac=1, random_state=0)
chosen['code'] = [f'P{i:02d}' for i in range(1, 41)]

# Private answer key (code -> ECG ID -> label), saved on Drive
key = chosen[['code', 'label']].reset_index()
key.to_csv(os.path.join(PROJECT_DIR, 'phone_photo_key.csv'), index=False)

# PDF: one ECG per page, filling the page, with only the code in the corner
pdf_path = os.path.join(PROJECT_DIR, 'ECGs_for_phone_photos.pdf')
with PdfPages(pdf_path) as pdf:
    for ecg_id, row in chosen.iterrows():
        fig = plt.figure(figsize=(11.69, 8.27))              # A4 landscape
        ax = fig.add_axes([0.02, 0.02, 0.96, 0.90])           # the ECG fills almost the whole page
        ax.imshow(Image.open(f'{IMG_DIR}/{ecg_id}.png'))
        ax.axis('off')
        fig.text(0.02, 0.95, row['code'], fontsize=22, weight='bold')
        pdf.savefig(fig, dpi=200)
        plt.close(fig)

print('Saved', pdf_path)
print(key['label'].value_counts())        # should show 20 of each
files.download(pdf_path)

Saved /content/drive/MyDrive/ECG_Vision_App/ECGs_for_phone_photos.pdf
label
0    20
1    20
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
from PIL import ImageFilter

def trace_pattern(path, size=(256, 128)):
    """Keep the dark trace lines, crop to them, shrink and blur: a simple picture of where the spikes are."""
    g = np.asarray(Image.open(path).convert('L'), dtype=np.float32)       # greyscale
    dark = g < np.percentile(g, 4)                                          # darkest 4% of pixels = the trace
    rows, cols = np.where(dark)
    r0, r1 = np.percentile(rows, [0.5, 99.5]).astype(int)                  # crop to where the trace is
    c0, c1 = np.percentile(cols, [0.5, 99.5]).astype(int)
    crop = Image.fromarray((dark[r0:r1, c0:c1] * 255).astype(np.uint8))
    small = crop.resize(size).filter(ImageFilter.GaussianBlur(2))          # small blur tolerates slight tilt
    v = np.asarray(small, dtype=np.float32).ravel()
    v = (v - v.mean()) / (v.std() + 1e-6)
    return v / np.sqrt(len(v))

PHOTO_DIR = os.path.join(PROJECT_DIR, 'phone_photos_cropped')
photos = sorted(f for f in os.listdir(PHOTO_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
print('Photos found:', len(photos))

orig = np.stack([trace_pattern(f'{IMG_DIR}/{i}.png') for i in chosen.index])
phot = np.stack([trace_pattern(os.path.join(PHOTO_DIR, f)) for f in photos])

sim = phot @ orig.T                                    # correlation of every photo with every original
order = np.argsort(-sim, axis=1)
match = pd.DataFrame({'photo': photos,
                      'matched_code': chosen['code'].values[order[:, 0]],
                      'similarity': sim[np.arange(len(photos)), order[:, 0]].round(3),
                      'margin_over_2nd': (sim[np.arange(len(photos)), order[:, 0]]
                                          - sim[np.arange(len(photos)), order[:, 1]]).round(3)})
print(match.to_string(index=False))

counts = match['matched_code'].value_counts()
print('\nCodes matched by 2+ photos:', list(counts[counts > 1].index))
print('Codes with no photo:', sorted(set(chosen['code']) - set(match['matched_code'])))

Photos found: 41
       photo matched_code  similarity  margin_over_2nd
IMG_7399.JPG          P01       0.638            0.032
IMG_7400.JPG          P02       0.730            0.146
IMG_7401.JPG          P03       0.687            0.136
IMG_7402.JPG          P04       0.750            0.222
IMG_7403.JPG          P10       0.294            0.048
IMG_7404.JPG          P06       0.691            0.084
IMG_7405.JPG          P07       0.598            0.162
IMG_7406.JPG          P11       0.287            0.003
IMG_7407.JPG          P09       0.369            0.062
IMG_7408.JPG          P10       0.595            0.184
IMG_7409.JPG          P11       0.614            0.136
IMG_7410.JPG          P29       0.304            0.036
IMG_7411.JPG          P13       0.486            0.047
IMG_7412.JPG          P38       0.528            0.004
IMG_7413.JPG          P29       0.341            0.013
IMG_7414.JPG          P16       0.423            0.000
IMG_7415.JPG          P10       0.215           

In [ ]:
# Match photos to codes using the ORDER they were taken in (codes can only stay the same or go up)
n_p, n_c = sim.shape
REPEAT, SKIP = 0.15, 0.10                  # small penalties: re-using a code (retake), skipping a code
score = np.full((n_p, n_c), -1e9)
back = np.zeros((n_p, n_c), dtype=int)
score[0] = sim[0] - SKIP * np.arange(n_c)
for i in range(1, n_p):
    for j in range(n_c):
        stay = score[i-1, j] - REPEAT                                    # same code as previous photo
        prev = [score[i-1, k] - SKIP * (j - k - 1) for k in range(j)]   # an earlier code
        best_k = int(np.argmax(prev)) if prev else -1
        if prev and prev[best_k] > stay:
            score[i, j], back[i, j] = prev[best_k] + sim[i, j], best_k
        else:
            score[i, j], back[i, j] = stay + sim[i, j], j
path = [int(np.argmax(score[-1]))]
for i in range(n_p - 1, 0, -1):
    path.append(back[i, path[-1]])
path = path[::-1]

aligned = pd.DataFrame({'photo': photos, 'code': chosen['code'].values[path],
                        'similarity': sim[np.arange(n_p), path].round(3)})
# If two photos got the same code (a retake), keep the one that matches best; the other is an extra
aligned['keep'] = aligned.groupby('code')['similarity'].transform('max') == aligned['similarity']
print(aligned.to_string(index=False))
print('\nExtras:', list(aligned.loc[~aligned['keep'], 'photo']))
print('Codes with no photo:', sorted(set(chosen['code']) - set(aligned.loc[aligned['keep'], 'code'])))

# Side-by-side check picture: each kept photo next to the original ECG it was matched to
kept = aligned[aligned['keep']].reset_index(drop=True)
fig, axes = plt.subplots(len(kept), 2, figsize=(10, 2.2 * len(kept)))
for r, row in kept.iterrows():
    ecg_id = chosen.index[chosen['code'] == row['code']][0]
    axes[r, 0].imshow(Image.open(os.path.join(PHOTO_DIR, row['photo'])).resize((496, 240)))
    axes[r, 1].imshow(Image.open(f'{IMG_DIR}/{ecg_id}.png').resize((496, 240)))
    axes[r, 0].set_title(row['photo'], fontsize=9); axes[r, 1].set_title(row['code'], fontsize=9)
    axes[r, 0].axis('off'); axes[r, 1].axis('off')
plt.tight_layout()
check_path = os.path.join(PROJECT_DIR, 'photo_match_check.jpg')
plt.savefig(check_path, dpi=50)
files.download(check_path)

In [7]:
# torch/torchvision: to rebuild ResNet-18 and load your trained weights
import torch, torch.nn as nn
from torchvision import models, transforms
from sklearn.metrics import confusion_matrix, f1_score, recall_score

# The matched photos, minus the 3 extras and the 2 pairs that looked wrong in the visual check
pairs = aligned[aligned['keep'] & ~aligned['photo'].isin(['IMG_7423.JPG', 'IMG_7428.JPG'])].copy()
pairs['label'] = [int(chosen.loc[chosen['code'] == c, 'label'].iloc[0]) for c in pairs['code']]
print('Photos scored:', len(pairs), '| AF:', pairs['label'].sum(), '| Normal:', (pairs['label'] == 0).sum())

# Same preparation as in training
to_model = transforms.Compose([transforms.Resize((240, 496)), transforms.ToTensor(),
                               transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
batch = torch.stack([to_model(Image.open(os.path.join(PHOTO_DIR, p)).convert('RGB')) for p in pairs['photo']])

def af_probs(weights_file):
    """Rebuild ResNet-18, load one of your saved models, and return its AF probability for each photo."""
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 1)
    m.load_state_dict(torch.load(os.path.join(PROJECT_DIR, 'models', weights_file), map_location='cpu'))
    m.eval()
    with torch.no_grad():
        return torch.sigmoid(m(batch).squeeze(1)).numpy()

pairs['old_prob'] = af_probs('resnet18_best.pt').round(2)            # first model (clean training)
pairs['new_prob'] = af_probs('resnet18_augmented_best.pt').round(2)  # augmented model

# Score each model at its own threshold (chosen earlier on validation, not changed now)
for name, col, t in [('OLD model (threshold 0.30)', 'old_prob', 0.30), ('NEW model (threshold 0.45)', 'new_prob', 0.45)]:
    preds = (pairs[col] >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(pairs['label'], preds, labels=[0, 1]).ravel()
    print(f'\n{name}: missed AF {fn} | false alarms {fp} | '
          f'sensitivity {recall_score(pairs["label"], preds):.3f} | '
          f'specificity {recall_score(pairs["label"], preds, pos_label=0):.3f} | F1 {f1_score(pairs["label"], preds):.3f}')

print('\n', pairs[['photo', 'code', 'label', 'old_prob', 'new_prob']].to_string(index=False))

Photos scored: 36 | AF: 17 | Normal: 19

OLD model (threshold 0.30): missed AF 0 | false alarms 1 | sensitivity 1.000 | specificity 0.947 | F1 0.971

NEW model (threshold 0.45): missed AF 2 | false alarms 0 | sensitivity 0.882 | specificity 1.000 | F1 0.938

        photo code  label  old_prob  new_prob
IMG_7399.JPG  P01      0      0.03      0.00
IMG_7400.JPG  P02      0      0.04      0.00
IMG_7401.JPG  P03      0      0.02      0.00
IMG_7402.JPG  P04      1      0.95      0.99
IMG_7403.JPG  P05      1      0.67      1.00
IMG_7404.JPG  P06      1      1.00      0.99
IMG_7405.JPG  P07      0      0.07      0.00
IMG_7406.JPG  P08      1      0.68      0.18
IMG_7407.JPG  P09      1      0.85      0.79
IMG_7408.JPG  P10      0      0.02      0.00
IMG_7409.JPG  P11      0      0.07      0.00
IMG_7410.JPG  P12      0      0.07      0.00
IMG_7411.JPG  P13      0      0.03      0.00
IMG_7412.JPG  P14      1      1.00      1.00
IMG_7413.JPG  P15      0      0.10      0.00
IMG_7414.JPG  P16   